In [ ]:
# ============================================================
# Google Colab setup — clone repo, install deps, stage dataset
# (Skip / comment out this cell when running locally.)
# ============================================================

# 1. Clone repository (Model branch) & install dependencies
!git clone -b Model https://REMOVED-TOKEN@github.com/Rockeyxx/Signature-Verification-ML-EE499.git /content/repo

%cd /content/repo
!pip install -q -r requirements.txt

# 2. Mount Google Drive (where your datasets live)
from google.colab import drive
drive.mount('/content/drive')

# 3. Copy the ZIPPED dataset to Colab's local SSD, then unzip.
#    Training reads every signature thousands of times; served straight off the
#    Drive FUSE mount that I/O is the bottleneck. Staging one .zip on the local
#    SSD and unzipping makes training ~10-20x faster.
!rm -rf /content/repo/data /content/data
!cp "/content/drive/MyDrive/ee499_data.zip" /content/ee499_data.zip   # change to your actual .zip path
!unzip -q /content/ee499_data.zip -d /content/repo/data
# After unzip, /content/repo/data should directly contain the dataset folders
# (cedar/, bhsig260/, utsig/, svc2004_task2/, ...). If your zip wraps them in a
# top-level folder, move them up or point DATA_DIR at that subfolder.
!ls /content/repo/data


# Signature Verification System — Interactive Pipeline

End-to-end **spatial + temporal** workflow for the Multimodal Signature Verification
System (EE 499).

Every heavy operation in this notebook is **imported from `scripts/`, not
re-implemented**: the training loop is `scripts.train.fit`, the metrics are
`scripts.evaluate.compute_eer` / `compute_far_frr`, and the ROC/DET/confusion
plots are `scripts.evaluate.plot_*`. Running this notebook and running
`python scripts/train.py` execute the same code, so the two cannot drift apart.

**Order of operations**

1. **Setup** — clone + install (Colab only), then imports and device selection
2. **Data** — discover datasets, preview genuine vs skilled-forgery pairs
3. **Split** — ISO/IEC 23053 writer-independent partition, then pairing
4. **Train** — spatial CNN, with live `tqdm` progress and a loss curve after
5. **Evaluate** — EER / FAR / FRR, confusion matrix, ROC and DET curves
6. **Temporal** — SVC2004 BiLSTM branch, same train/evaluate path
7. **Fuse** — late-fusion decision engine across both branches


In [ ]:
import os
import sys
import random

import numpy as np
import cv2
import torch
import torch.optim as optim
import matplotlib.pyplot as plt

# Add project root directory to path
sys.path.insert(0, os.path.abspath('.'))

from config.settings import (
    SPATIAL_EMBEDDING_DIM, TEMPORAL_EMBEDDING_DIM,
    CONTRASTIVE_MARGIN, LEARNING_RATE, BATCH_SIZE,
    NUM_EPOCHS, WEIGHT_DECAY, MODEL_WEIGHTS_DIR,
    DECISION_THRESHOLD, TARGET_FAR, TARGET_FRR,
    HARD_NEGATIVE_RATIO, DATA_DIR,
)
from src.models.siamese import SpatialSiamese, TemporalSiamese, ContrastiveLoss, FusionEngine
from src.data.dataset import preprocess_image, writer_independent_split, generate_skilled_pairs
from src.data.parsers import parse_dataset

# ── Reused pipeline code ────────────────────────────────────────────────────
# Imported, never re-implemented. `fit` is the same epoch loop (checkpointing,
# best-weight saving, early stopping) that `python scripts/train.py` runs, and
# the metric/plot helpers are the ones `scripts/evaluate.py` uses. tqdm bars
# come from inside those functions, so they appear here and on the CLI alike.
from scripts.train import build_loader, fit
from scripts.evaluate import (
    score_pairs, compute_eer, compute_far_frr,
    plot_loss_curve, plot_roc, plot_det, plot_confusion_matrix,
)

# Random seeds for reproducible evaluation
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# DataLoader worker processes — must be 0 in notebooks (Colab and Windows both
# break on forked workers inside a kernel).
NUM_WORKERS = 0

# Dynamic hardware device selection
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Hardware Device Selected: {device}")

# Clear GPU memory cache to prevent Out-Of-Memory (OOM) errors
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("VRAM memory cache cleared successfully.")


In [ ]:
# Load real datasets through the dataset parsers.
#
# The previous os.walk() discovery treated any directory containing images as a
# single "writer", which would have collapsed all 55 CEDAR signers into one
# entry (data/cedar/full_org) and silently destroyed writer independence. It
# also filtered on .png/.jpg only, skipping the .tif files BHSig260 and UTSig
# ship. The parsers handle each dataset's real on-disk layout instead.
SPATIAL_DATASETS = ["cedar", "bhsig260", "utsig"]

samples_by_writer = {}
for name in SPATIAL_DATASETS:
    path = os.path.join(DATA_DIR, name)
    if os.path.isdir(path):
        parsed = parse_dataset(path)
        samples_by_writer.update(parsed)
        print(f"{name}: {len(parsed)} writers")

print(f"Total: {len(samples_by_writer)} writers")

# Fallback: generate a small synthetic dataset so the notebook runs without data
if not samples_by_writer:
    print("No datasets found under ./data. Generating synthetic signature pairs for demonstration...")
    demo_root = os.path.join(DATA_DIR, "demo_dataset")
    os.makedirs(demo_root, exist_ok=True)

    for writer in range(1, 5):
        genuine, forged = [], []
        for i in range(1, 6):
            img = np.ones((256, 256), dtype=np.uint8) * 255
            cv2.line(img, (30 + i * 5, 128), (220, 128 + writer * 4), 0, 3)
            path = f"{demo_root}/w{writer}_g{i}.png"
            cv2.imwrite(path, img)
            genuine.append(path)

            img = np.ones((256, 256), dtype=np.uint8) * 255
            cv2.line(img, (34 + i * 5, 132), (216, 124 + writer * 4), 0, 3)
            path = f"{demo_root}/w{writer}_f{i}.png"
            cv2.imwrite(path, img)
            forged.append(path)

        samples_by_writer[f"demo_{writer}"] = {"genuine": genuine, "forged": forged}
    print(f"Demo dataset created with {len(samples_by_writer)} writers.")

# Generate sample pairs for preview visualization (skilled-forgery negatives)
demo_pairs, demo_labels = generate_skilled_pairs(
    samples_by_writer, num_genuine_per_writer=2, num_forged_per_writer=2)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle("Dataset Sample Preview — Genuine vs Skilled-Forgery Pairs", fontsize=14, fontweight="bold")

preview = list(zip(demo_pairs, demo_labels))
random.Random(42).shuffle(preview)

for ax, ((p1, p2), label) in zip(axes.flat, preview[:8]):
    img1 = preprocess_image(p1)[0]
    img2 = preprocess_image(p2)[0]
    ax.imshow(np.hstack([img1, img2]), cmap="gray")
    status = "Genuine Pair (y=1)" if label == 1 else "Skilled Forgery (y=0)"
    ax.set_title(status, color="green" if label == 1 else "red", fontsize=10, fontweight="bold")
    ax.axis("off")

plt.tight_layout()
os.makedirs("./results", exist_ok=True)
plt.savefig("./results/dataset_samples.png", dpi=150)
plt.show()


In [ ]:
# Perform ISO/IEC 23053 writer-independent data partition split
writer_ids = sorted(samples_by_writer.keys())
train_ids, val_ids, test_ids = writer_independent_split(
    writer_ids, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, seed=SEED)

print(f"Writer Split Ratios: {len(train_ids)} Train | {len(val_ids)} Val | {len(test_ids)} Test")

# build_loader() is scripts/train.py's loader factory. It generates pairs AFTER
# the writer split (so no signer straddles a partition boundary) and uses one
# seed for every split — the same seed the CLI uses, so weights trained here and
# scored by scripts/evaluate.py see byte-identical pairs.
train_loader, n_train = build_loader(samples_by_writer, train_ids, "spatial",
                                     BATCH_SIZE, NUM_WORKERS, True, SEED)
val_loader, n_val = build_loader(samples_by_writer, val_ids or train_ids, "spatial",
                                 BATCH_SIZE, NUM_WORKERS, False, SEED)
test_loader, n_test = build_loader(samples_by_writer, test_ids or train_ids, "spatial",
                                   BATCH_SIZE, NUM_WORKERS, False, SEED)

print(f"Generated Pairs: {n_train} Train | {n_val} Val | {n_test} Test")


In [ ]:
# ============================================================
# Spatial branch — train (or load) SpatialSiamese
# ============================================================
# The epoch loop, per-epoch checkpointing, best-validation weight saving and
# early stopping all live in scripts/train.py::fit. This cell only decides
# whether to train, and plots the history fit() hands back.
weights_path = os.path.join(MODEL_WEIGHTS_DIR, "spatial_cnn.pth")
checkpoint_path = os.path.join(MODEL_WEIGHTS_DIR, "checkpoint_spatial_cnn.pth")
FORCE_TRAIN = False  # Set to True to force re-training

model = SpatialSiamese(embedding_dim=SPATIAL_EMBEDDING_DIM).to(device)
loss_fn = ContrastiveLoss(margin=CONTRASTIVE_MARGIN, hard_negative_ratio=HARD_NEGATIVE_RATIO)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

history = None
if os.path.exists(weights_path) and not FORCE_TRAIN:
    print(f"Pre-compiled model weights found at: {weights_path}")
    model.load_state_dict(torch.load(weights_path, map_location=device))
    print("Weights loaded successfully into model. Skipping training phase.")
else:
    print(f"Starting training phase for {NUM_EPOCHS} epochs...")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    history = fit(model, train_loader, val_loader, loss_fn, optimizer, device,
                  NUM_EPOCHS, weights_path, checkpoint_path)

    print(f"Training complete. Best model weights saved to: {weights_path}")

# Loss curve — how training and validation loss changed over the epochs just run.
# Drawn after fit() returns so the tqdm bars above stay intact. plot_loss_curve
# is shared with the temporal branch and marks the best-validation epoch, which
# is the one whose weights were actually saved.
if history:
    fig, ax = plt.subplots(figsize=(8, 5))
    plot_loss_curve(history, "Spatial CNN — Training vs Validation Loss", ax=ax)
    fig.tight_layout()
    os.makedirs("./results", exist_ok=True)
    fig.savefig("./results/training_loss.png", dpi=150)
    plt.show()
else:
    print("Loaded pre-trained weights — no loss curve to draw. "
          "Set FORCE_TRAIN = True to retrain and see it.")


In [ ]:
# ============================================================
# Spatial branch — evaluation (EER / FAR / FRR)
# ============================================================
# score_pairs / compute_eer / compute_far_frr are imported from
# scripts/evaluate.py — identical numbers to the CLI, given the same weights.
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# tau is tuned on the VALIDATION split only (AGENTS.md). Choosing it on test
# would report the best-case threshold rather than an honest operating point.
val_scores, val_lbl = score_pairs(model, val_loader, device, desc="spatial · val (tune τ)")
_, tau_val = compute_eer(val_scores[val_lbl == 1], val_scores[val_lbl == 0])

# score_pairs returns scores in loader order; the loader is shuffle-free, so
# scores[i] corresponds to test_loader.dataset.pairs[i] (used by the grid below).
scores, all_labels = score_pairs(model, test_loader, device, desc="spatial · test")
genuine_scores = scores[all_labels == 1]
impostor_scores = scores[all_labels == 0]

eer, eer_threshold = compute_eer(genuine_scores, impostor_scores)
far_tuned, frr_tuned = compute_far_frr(genuine_scores, impostor_scores, tau_val)
far_fixed, frr_fixed = compute_far_frr(genuine_scores, impostor_scores, DECISION_THRESHOLD)

predictions = (scores >= tau_val).astype(int)
accuracy = np.mean(predictions == all_labels)

print("=" * 66)
print("                  SPATIAL EVALUATION REPORT")
print("=" * 66)
print(f"  Test pairs:            {len(scores)}  "
      f"({len(genuine_scores)} genuine / {len(impostor_scores)} impostor)")
print(f"  Test Accuracy:         {accuracy * 100:.2f}%   (at τ_val)")
print(f"  Equal Error Rate:      {eer * 100:.2f}%   (τ @ EER: {eer_threshold:.4f})")
print("-" * 66)
print(f"  {'':<28}{'τ_val':>12}{'τ_fixed':>14}")
print(f"  {'threshold':<28}{tau_val:>12.4f}{DECISION_THRESHOLD:>14.4f}")
print(f"  {'False Acceptance (FAR) %':<28}{far_tuned * 100:>12.2f}{far_fixed * 100:>14.2f}"
      f"   target <{TARGET_FAR * 100:.0f}%")
print(f"  {'False Rejection (FRR) %':<28}{frr_tuned * 100:>12.2f}{frr_fixed * 100:>14.2f}"
      f"   target <{TARGET_FRR * 100:.0f}%")
print("=" * 66)
print("  τ_val  = tuned on the validation split, per AGENTS.md")
print("  τ_fixed = config DECISION_THRESHOLD, the value the GUI ships with")


In [ ]:
# ============================================================
# Spatial branch — evaluation plots
# ============================================================
# plot_confusion_matrix / plot_roc / plot_det are the same functions
# scripts/evaluate.py writes into results/. Passing ax= draws them inline here
# instead of saving, so one implementation serves the CLI and the notebook.
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plot_confusion_matrix(genuine_scores, impostor_scores, tau_val,
                      f"Confusion Matrix (τ={tau_val:.3f})", ax=axes[0])
plot_roc(genuine_scores, impostor_scores, f"ROC Curve (EER={eer * 100:.2f}%)", ax=axes[1])
plot_det(genuine_scores, impostor_scores, "DET Curve", ax=axes[2])
plt.tight_layout()
os.makedirs("./results", exist_ok=True)
plt.savefig("./results/evaluation_plots.png", dpi=150)
plt.show()

# Sample predictions grid — notebook-only, no CLI equivalent to reuse.
# Safe because scores/all_labels are in loader order and the loader never shuffles.
test_pairs = test_loader.dataset.pairs

fig, axes = plt.subplots(2, 4, figsize=(15, 7))
fig.suptitle("Sample Model Predictions vs Ground Truth", fontsize=14, fontweight="bold")

for idx, (ax, (p1, p2)) in enumerate(zip(axes.flat, test_pairs[:8])):
    # preprocess_image gives the fixed-size view the model actually saw, and
    # sidesteps hstack failing on two source images of different dimensions.
    img1 = preprocess_image(p1)[0]
    img2 = preprocess_image(p2)[0]
    ax.imshow(np.hstack([img1, img2]), cmap="gray")

    gt = "Genuine" if all_labels[idx] == 1 else "Forged"
    pred = "Genuine" if predictions[idx] == 1 else "Forged"
    ax.set_title(f"GT: {gt} | Pred: {pred}\nScore: {scores[idx] * 100:.1f}%",
                 color="green" if gt == pred else "red", fontsize=9, fontweight="bold")
    ax.axis("off")

plt.tight_layout()
plt.savefig("./results/sample_predictions.png", dpi=150)
plt.show()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("Spatial branch complete. VRAM cache cleared.")


## Temporal Branch (SVC2004) + Multimodal Fusion

The cells above cover the **spatial** branch (offline signature *images*). This
section adds the **temporal** branch — online, dynamic signatures from **SVC2004**,
where each sample is a pen trajectory of `(X, Y, Pressure)` points — and then
demonstrates the **late-fusion** decision engine.

1. **Temporal data** — parse SVC2004, writer-independent split, variable-length
   trajectory pairs (padded + packed via `collate_temporal_pairs`).
2. **Temporal training** — `TemporalSiamese` (BiLSTM) → `temporal_lstm.pth`.
3. **Temporal evaluation** — EER / FAR / FRR on the held-out temporal writers.
4. **Fusion demo** — `FusionEngine` combining spatial + temporal scores, with
   unimodal fallback and degraded-mode routing.

> **Note on fusion scope.** The offline image datasets and SVC2004 are *disjoint* —
> no single signing event exists as both a rendered image **and** a trajectory —
> so there is no paired multimodal test set to benchmark end-to-end. The fusion
> cell therefore demonstrates the real `FusionEngine` **decision logic** (α-weighting,
> routing, degraded flags) on each trained branch's actual score magnitudes.

In [ ]:
# ============================================================
# Temporal branch — load SVC2004 online (dynamic) signatures
# ============================================================
from config.settings import (
    SVC2004_DIR, TEMPORAL_HIDDEN_SIZE, TEMPORAL_NUM_LAYERS, PRESSURE_DROPOUT_RATE,
)
from src.data.parsers import svc2004_has_pressure

svc_root = SVC2004_DIR
# Sentinels so downstream cells never reference an unbound name when SVC2004
# is absent (every consumer is still guarded by `if TEMPORAL_AVAILABLE`).
TEMPORAL_AVAILABLE = False
temporal_writer_samples = {}
has_pressure = False
t_pressure_dropout = 0.0
temporal_weights_name = "temporal_lstm.pth"
t_train_loader = t_val_loader = t_test_loader = None

try:
    temporal_writer_samples = parse_dataset(svc_root)   # {writer_id: [trajectory_sample, ...]}
    TEMPORAL_AVAILABLE = len(temporal_writer_samples) > 0
except Exception as exc:
    print(f"SVC2004 not found at '{svc_root}' ({exc}).")

if not TEMPORAL_AVAILABLE:
    print("Skipping the temporal branch. Point `data/svc2004_task2` (or task1) at "
          "the SVC2004 dataset to enable it.")
else:
    print(f"SVC2004 writers discovered: {len(temporal_writer_samples)}")

    # Task2 (7-col) has real pressure -> pressure dropout is meaningful.
    # Task1 (4-col) imputes P=1.0 -> dropout is a no-op (pressure-blind ablation).
    has_pressure = svc2004_has_pressure(svc_root)
    t_pressure_dropout = PRESSURE_DROPOUT_RATE if has_pressure else 0.0
    temporal_weights_name = "temporal_lstm.pth" if has_pressure else "temporal_lstm_task1.pth"
    print(f"Pressure channel: {'present (Task2)' if has_pressure else 'imputed (Task1 ablation)'} "
          f"| dropout={t_pressure_dropout:.0%} | weights={temporal_weights_name}")

    # Writer-independent split BEFORE pairing (same ISO/IEC 23053 protocol as spatial).
    t_writer_ids = sorted(temporal_writer_samples.keys())
    t_train_ids, t_val_ids, t_test_ids = writer_independent_split(
        t_writer_ids, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, seed=SEED)
    print(f"Temporal split: {len(t_train_ids)} train | {len(t_val_ids)} val | "
          f"{len(t_test_ids)} test writers")

    # Same build_loader as the spatial branch: branch="temporal" swaps in the
    # padding collate that variable-length trajectories need.
    t_train_loader, t_n_train = build_loader(temporal_writer_samples, t_train_ids, "temporal",
                                             BATCH_SIZE, NUM_WORKERS, True, SEED)
    t_val_loader, t_n_val = build_loader(temporal_writer_samples, t_val_ids or t_train_ids, "temporal",
                                         BATCH_SIZE, NUM_WORKERS, False, SEED)
    t_test_loader, t_n_test = build_loader(temporal_writer_samples, t_test_ids or t_train_ids, "temporal",
                                           BATCH_SIZE, NUM_WORKERS, False, SEED)
    print(f"Temporal pairs: {t_n_train} train | {t_n_val} val | {t_n_test} test")


In [ ]:
# ============================================================
# Temporal branch — train (or load) TemporalSiamese (BiLSTM)
# ============================================================
# Same scripts/train.py::fit as the spatial branch — fit() reads the batch shape
# and routes packed sequences automatically, so there is no separate loop here.
temporal_model = None
t_history = None

if not TEMPORAL_AVAILABLE:
    print("Temporal data unavailable — skipping temporal training.")
else:
    temporal_weights_path = os.path.join(MODEL_WEIGHTS_DIR, temporal_weights_name)
    temporal_checkpoint_path = os.path.join(
        MODEL_WEIGHTS_DIR, f"checkpoint_{os.path.splitext(temporal_weights_name)[0]}.pth")
    FORCE_TRAIN_TEMPORAL = False  # Set to True to force re-training

    temporal_model = TemporalSiamese(
        hidden_size=TEMPORAL_HIDDEN_SIZE,
        num_layers=TEMPORAL_NUM_LAYERS,
        embedding_dim=TEMPORAL_EMBEDDING_DIM,
        pressure_dropout=t_pressure_dropout,
    ).to(device)
    temporal_loss_fn = ContrastiveLoss(margin=CONTRASTIVE_MARGIN,
                                       hard_negative_ratio=HARD_NEGATIVE_RATIO)
    temporal_optimizer = optim.Adam(temporal_model.parameters(), lr=LEARNING_RATE,
                                    weight_decay=WEIGHT_DECAY)

    if os.path.exists(temporal_weights_path) and not FORCE_TRAIN_TEMPORAL:
        print(f"Pre-compiled temporal weights found at: {temporal_weights_path}")
        temporal_model.load_state_dict(torch.load(temporal_weights_path, map_location=device))
        print("Weights loaded successfully. Skipping temporal training phase.")
    else:
        print(f"Starting temporal training for {NUM_EPOCHS} epochs...")
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        t_history = fit(temporal_model, t_train_loader, t_val_loader, temporal_loss_fn,
                        temporal_optimizer, device, NUM_EPOCHS,
                        temporal_weights_path, temporal_checkpoint_path)

        print(f"Temporal training complete. Best weights saved to: {temporal_weights_path}")

if t_history:
    fig, ax = plt.subplots(figsize=(8, 5))
    plot_loss_curve(t_history, "Temporal BiLSTM — Training vs Validation Loss", ax=ax)
    fig.tight_layout()
    os.makedirs("./results", exist_ok=True)
    fig.savefig("./results/temporal_training_loss.png", dpi=150)
    plt.show()


In [ ]:
# ============================================================
# Temporal branch — evaluation (EER / FAR / FRR)
# ============================================================
t_genuine_scores = np.array([])
t_impostor_scores = np.array([])

if not TEMPORAL_AVAILABLE:
    print("Temporal data unavailable — skipping temporal evaluation.")
else:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Identical protocol to the spatial branch: tune τ on validation, report on test.
    t_val_scores, t_val_lbl = score_pairs(temporal_model, t_val_loader, device,
                                          desc="temporal · val (tune τ)")
    _, t_tau_val = compute_eer(t_val_scores[t_val_lbl == 1], t_val_scores[t_val_lbl == 0])

    t_scores, t_all_labels = score_pairs(temporal_model, t_test_loader, device,
                                         desc="temporal · test")
    t_genuine_scores = t_scores[t_all_labels == 1]
    t_impostor_scores = t_scores[t_all_labels == 0]

    t_eer, t_eer_threshold = compute_eer(t_genuine_scores, t_impostor_scores)
    t_far_tuned, t_frr_tuned = compute_far_frr(t_genuine_scores, t_impostor_scores, t_tau_val)
    t_far_fixed, t_frr_fixed = compute_far_frr(t_genuine_scores, t_impostor_scores,
                                               DECISION_THRESHOLD)
    t_accuracy = np.mean((t_scores >= t_tau_val).astype(int) == t_all_labels)

    print("=" * 66)
    print("                 TEMPORAL EVALUATION REPORT")
    print("=" * 66)
    print(f"  Test pairs:            {len(t_scores)}  "
          f"({len(t_genuine_scores)} genuine / {len(t_impostor_scores)} impostor)")
    print(f"  Test Accuracy:         {t_accuracy * 100:.2f}%   (at τ_val)")
    print(f"  Equal Error Rate:      {t_eer * 100:.2f}%   (τ @ EER: {t_eer_threshold:.4f})")
    print("-" * 66)
    print(f"  {'':<28}{'τ_val':>12}{'τ_fixed':>14}")
    print(f"  {'threshold':<28}{t_tau_val:>12.4f}{DECISION_THRESHOLD:>14.4f}")
    print(f"  {'False Acceptance (FAR) %':<28}{t_far_tuned * 100:>12.2f}"
          f"{t_far_fixed * 100:>14.2f}   target <{TARGET_FAR * 100:.0f}%")
    print(f"  {'False Rejection (FRR) %':<28}{t_frr_tuned * 100:>12.2f}"
          f"{t_frr_fixed * 100:>14.2f}   target <{TARGET_FRR * 100:.0f}%")
    print("=" * 66)

    # Same three plot helpers, drawn inline via ax=.
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    plot_confusion_matrix(t_genuine_scores, t_impostor_scores, t_tau_val,
                          f"Temporal Confusion (τ={t_tau_val:.3f})", ax=axes[0])
    plot_roc(t_genuine_scores, t_impostor_scores,
             f"Temporal ROC (EER={t_eer * 100:.2f}%)", ax=axes[1])
    plot_det(t_genuine_scores, t_impostor_scores, "Temporal DET", ax=axes[2])
    plt.tight_layout()
    os.makedirs("./results", exist_ok=True)
    plt.savefig("./results/temporal_evaluation_plots.png", dpi=150)
    plt.show()


In [ ]:
# ============================================================
# Multimodal late fusion — FusionEngine decision logic
# ============================================================
# S_final = alpha * S_spatial + (1 - alpha) * S_temporal, with automatic
# fallback to whichever branch is present. Because the datasets are disjoint
# (no signer has both an image and a trajectory), we exercise the engine on
# each branch's *representative* genuine / impostor scores rather than a paired
# test set — this validates the routing and degraded-mode contract the GUI renders.
from config.settings import ALPHA_DEFAULT

engine = FusionEngine(alpha=ALPHA_DEFAULT, threshold=DECISION_THRESHOLD)

have_spatial = 'genuine_scores' in dir() and len(genuine_scores) > 0
have_temporal = TEMPORAL_AVAILABLE and len(t_genuine_scores) > 0

if not (have_spatial or have_temporal):
    print("Need at least one evaluated branch to demonstrate fusion.")
else:
    # Representative operating points: median genuine (should pass) and median
    # impostor (should fail) similarity for each available branch.
    s_gen = float(np.median(genuine_scores)) if have_spatial else None
    s_imp = float(np.median(impostor_scores)) if have_spatial else None
    t_gen = float(np.median(t_genuine_scores)) if have_temporal else None
    t_imp = float(np.median(t_impostor_scores)) if have_temporal else None

    print(f"Fusion alpha (spatial weight): {ALPHA_DEFAULT}  |  decision threshold: {DECISION_THRESHOLD}")
    print(f"Representative scores  ->  spatial: gen={s_gen}, imp={s_imp}  |  temporal: gen={t_gen}, imp={t_imp}")
    print("-" * 60)

    def show(title, **kw):
        print(f"{title}")
        print(f"   {engine.fuse(**kw)}")

    if have_spatial and have_temporal:
        show("Multimodal  — genuine spatial + genuine temporal:", spatial_score=s_gen, temporal_score=t_gen)
        show("Multimodal  — impostor spatial + impostor temporal:", spatial_score=s_imp, temporal_score=t_imp)
        show("Multimodal  — imputed pressure (degraded flag):",
             spatial_score=s_gen, temporal_score=t_gen, pressure_imputed=True)
    if have_spatial:
        show("Spatial-only fallback (temporal branch missing):", spatial_score=s_gen)
    if have_temporal:
        show("Temporal-only fallback (spatial branch missing):", temporal_score=t_gen)